# Compat Truth Table

Exercises compatibility report setup and documents how moved aliases are protected outside strict notebook coverage.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.io.list_logs",
    "tl.io.load",
    "tl.io.load_intervention_spec",
    "tl.io.log_model_metadata",
    "tl.io.rehydrate_nested",
    "tl.io.reset_naming_counter",
    "tl.io.save",
    "tl.io.save_intervention",
    "tl.io.suppress_mutate_warnings",
    "tl.label",
    "tl.llm",
    "tl.load",
    "tl.log_forward_pass",
    "tl.mean_ablate",
    "tl.module",
    "tl.multi_trace",
    "tl.multi_trace.METRIC_REGISTRY",
    "tl.multi_trace.NodeView",
    "tl.multi_trace.Supergraph",
    "tl.multi_trace.SupergraphNode",
    "tl.multi_trace.TopologyDiff",
    "tl.multi_trace.build_supergraph",
    "tl.multi_trace.compare_topology",
    "tl.multi_trace.cosine_distance",
    "tl.multi_trace.pearson_correlation_distance",
    "tl.multi_trace.relative_l1_scalar",
    "tl.multi_trace.relative_l2",
    "tl.multi_trace.resolve_metric",
    "tl.neuro",
    "tl.noise",
    "tl.notebook",
    "tl.observers",
    "tl.observers.TapObserver",
    "tl.observers.TapRecord",
    "tl.observers.active_span_records",
    "tl.observers.record_span",
    "tl.observers.tap",
    "tl.options",
    "tl.options.CaptureOptions",
    "tl.options.InterventionOptions",
    "tl.options.ReplayOptions",
    "tl.options.SaveOptions",
    "tl.options.StreamingOptions",
    "tl.options.VisualizationOptions",
    "tl.options.merge_capture_options",
    "tl.options.merge_intervention_options",
    "tl.options.merge_replay_options",
    "tl.options.merge_save_options",
    "tl.options.merge_streaming_options",
    "tl.options.merge_visualization_options",
    "tl.options.suppress_mutate_warnings",
    "tl.options.visualization_to_render_kwargs",
    "tl.paper",
    "tl.partial",
    "tl.partial.PartialModelLog",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")